In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

scaler = StandardScaler()
scaler.fit(X_train)                 # mean/std SIRF train se seekha
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)   # test par sirf lagaya, seekha nahi

NameError: name 'X' is not defined

In [3]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# data (pehle wala hi, seed 7)
np.random.seed(7)
n = 300
free     = np.random.randint(0, 8, n)
links    = np.random.randint(0, 6, n)
caps     = np.random.randint(0, 10, n)
exclaim  = np.random.randint(0, 6, n)
score = 1.0*free + 0.8*links + 0.5*caps + 0.6*exclaim + np.random.randn(n)*3
y = (score > np.percentile(score, 65)).astype(int)
X = np.column_stack([free, links, caps, exclaim])

# split + scaler
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
scaler = StandardScaler()
scaler.fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

print(scaler.mean_)
print(X_train.mean(axis=0))

[3.60416667 2.51666667 4.52083333 2.4       ]
[3.60416667 2.51666667 4.52083333 2.4       ]


In [4]:
print(X_train_s.mean(axis=0).round(2))
print(X_test_s.mean(axis=0).round(2))

[0. 0. 0. 0.]
[ 0.03  0.01 -0.21  0.06]


In [5]:
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
scores = cross_val_score(pipe, X, y, cv=5, scoring="f1")
print(scores.round(2), scores.mean().round(2))

[0.67 0.59 0.73 0.65 0.61] 0.65


In [6]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_s, y_train)

names = ["free", "links", "caps", "exclaim"]
for name, w in zip(names, model.coef_[0]):
    print(name, round(w, 2))

free 1.52
links 1.08
caps 0.68
exclaim 0.63


In [7]:
for seed in range(5):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed)
    sc = StandardScaler().fit(Xtr)
    m = LogisticRegression().fit(sc.transform(Xtr), ytr)
    print(seed, "caps:", round(m.coef_[0][2], 2), "exclaim:", round(m.coef_[0][3], 2))

0 caps: 0.68 exclaim: 0.63
1 caps: 0.84 exclaim: 0.62
2 caps: 1.0 exclaim: 0.41
3 caps: 0.82 exclaim: 0.5
4 caps: 0.82 exclaim: 0.56


In [8]:
from sklearn.metrics import f1_score

for C in [0.001, 0.01, 1, 100]:
    m = LogisticRegression(C=C).fit(X_train_s, y_train)
    tr = f1_score(y_train, m.predict(X_train_s))
    te = f1_score(y_test,  m.predict(X_test_s))
    print("C =", C, "| train F1:", round(tr, 2), "| test F1:", round(te, 2))

C = 0.001 | train F1: 0.0 | test F1: 0.0
C = 0.01 | train F1: 0.43 | test F1: 0.4
C = 1 | train F1: 0.7 | test F1: 0.63
C = 100 | train F1: 0.7 | test F1: 0.63
